# Notebook 06: Foundations - Python Concepts for Deep Learning

---

## What This Notebook Covers

This notebook introduces several important Python programming concepts that are foundational for building deep learning frameworks. These concepts include:

1. **Callbacks** - A way to customize behavior of functions/classes without modifying their code
2. **Lambda functions** - Short, anonymous functions written in one line
3. **Partial functions** - Functions with some arguments pre-filled
4. **Callable classes** - Objects that can be called like functions
5. **`*args` and `**kwargs`** - Flexible ways to handle function arguments
6. **Dunder (double underscore) methods** - Special Python methods that enable custom behavior

**Why are these important?** Deep learning frameworks like PyTorch and fastai use these patterns extensively. Understanding them will help you:
- Customize training loops
- Add logging, early stopping, and other features
- Build flexible, reusable code

---

## 🎛️ Interactive visualizations in this notebook

This notebook is paired with **7 interactive visualizations** (plain HTML/JS — no internet or extra installs needed) that appear inline at the right moments. They all share the same control scheme:

| Control | What it does |
|---|---|
| **⏮ Prev / Next ⏭** | move one micro-step backward / forward |
| **▶ Play / ⏸ Pause** | run the whole animation automatically |
| **🐢 ── 🐇 speed slider** | slow it down to read, or speed it up to review |
| **scrubber** | drag to any step, like a video timeline |
| **click a stage** | epochs, timeline nodes, pipeline boxes, bars, flowchart boxes are *clickable* — jump straight to that moment |
| **purple panel** | a plain-English narration of exactly what the current step is doing |

Whenever you jump or step, the **corresponding line of code is highlighted**, so the animation and the code never drift apart.

| # | File | Concept |
|---|---|---|
| 1 | `callback_playground.html` | what a callback *is* (GUI + training loop) |
| 2 | `lambda_partial_machine.html` | lambdas, closures and `partial` as argument-filling machines |
| 3 | `args_kwargs_sorter.html` | `*args` / `**kwargs` packing and unpacking |
| 4 | `callback_hooks_timeline.html` | multi-hook callbacks, `hasattr` gates, early stopping |
| 5 | `state_modifying_calculator.html` | callbacks that *modify* the calculation's state |
| 6 | `dunder_dispatch.html` | `__dunder__` methods — how `a + b` becomes your code |
| 7 | `getattr_lookup_flow.html` | `__getattr__` and the attribute lookup walk |

Keep the folder `06_foundations_visuals/` next to this notebook.

## Setup: Importing Required Libraries

We start by importing the libraries we'll use throughout this notebook.

In [4]:
# =====================================================
# IMPORTS
# =====================================================

# PyTorch - the deep learning library (we'll use it later in the course)
import torch

# Matplotlib - for creating visualizations and plots
import matplotlib.pyplot as plt

# Random - Python's built-in module for generating random numbers
import random

---

# Part 1: Callbacks

---

## What is a Callback?

A **callback** is a function that you pass to another function, which will be "called back" (executed) at a specific point during that function's execution.

**Simple Analogy:** Think of ordering food at a restaurant:
- You tell the waiter "Call me when my food is ready" (you're registering a callback)
- The kitchen prepares your food (the main process runs)
- When food is ready, the waiter calls you (the callback is executed)

**Why use callbacks?**
- To customize behavior without changing the original function
- To get notified when something happens
- To inject custom code at specific points in a process

In deep learning, callbacks are used for:
- Printing progress during training
- Saving the model at checkpoints
- Early stopping when validation loss stops improving
- Logging metrics to visualization tools

---

## 1.1 Callbacks as GUI Events

The most familiar example of callbacks is in graphical user interfaces (GUIs). When you click a button, something happens - that "something" is defined by a callback function.

Let's see an example using Jupyter widgets (interactive elements in notebooks).

In [5]:
# Import ipywidgets - a library for creating interactive widgets in Jupyter
# Widgets are interactive elements like buttons, sliders, text boxes, etc.
import ipywidgets as widgets

From the [ipywidget docs](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20Events.html):

> *"The button widget is used to handle mouse clicks. The `on_click` method of the Button can be used to register a function to be called when the button is clicked."*

In other words:
- We create a button
- We tell the button "when you're clicked, run this function"
- The function we provide is the **callback**

In [6]:
# =====================================================
# CREATING A BUTTON WIDGET
# =====================================================

# widgets.Button() creates a clickable button
# 
# Arguments:
#   description - The text displayed on the button face
#
# The button is stored in variable 'w' (short for widget)

w = widgets.Button(description='Click me')

In [7]:
# Display the button in the notebook
# Simply writing a variable name at the end of a cell displays it
w

Button(description='Click me', style=ButtonStyle())

**What you see above:** A button labeled "Click me" should appear. If you click it now, nothing happens because we haven't told it what to do when clicked.

Now let's define what should happen when the button is clicked:

In [8]:
# =====================================================
# DEFINING A CALLBACK FUNCTION
# =====================================================

# This function will be called when the button is clicked.
# 
# Arguments:
#   o - The button widget object itself (passed automatically by the button)
#       We don't use it here, but it's required by the on_click mechanism
#
# What it does:
#   Simply prints 'hi' to the output

def f(o): 
    print('hi')

In [9]:
# =====================================================
# REGISTERING THE CALLBACK
# =====================================================

# on_click() tells the button: "When you're clicked, call this function"
#
# Arguments:
#   f - The function to call when button is clicked (our callback)
#
# IMPORTANT: We pass `f` not `f()`
#   - `f` is the function object itself (what we want)
#   - `f()` would CALL the function immediately (not what we want)

w.on_click(f)

**Now try clicking the button above!** Each time you click it, 'hi' should be printed.

---

**Key Concept: Passing Functions as Arguments**

Notice that we wrote `w.on_click(f)` and **not** `w.on_click(f())`:

| Code | What it means |
|------|---------------|
| `f` | The function object itself - can be passed around, stored, called later |
| `f()` | Call the function NOW and use its return value |

When we pass `f` to `on_click`, we're saying "here's the function, call it later when the button is clicked."

---

*NB: When callbacks are used in this way they are often called "events".* The button "click" is an event, and our function `f` is the event handler.

---

## 1.2 Creating Your Own Callback

Now let's see how to create our own functions that accept callbacks. This is more relevant to deep learning where we want to customize training behavior.

We'll create a "slow calculation" that simulates training epochs (iterations through data).

In [10]:
# Import sleep from the time module
# sleep(n) pauses execution for n seconds
# We use this to simulate a slow computation
from time import sleep

In [11]:
# =====================================================
# A SLOW CALCULATION (VERSION 1 - NO CALLBACK)
# =====================================================

def slow_calculation():
    """
    Simulates a slow computation that runs for 5 iterations.
    
    This is like a training loop that runs for 5 epochs.
    Each iteration takes 1 second (simulated by sleep).
    
    Returns:
        The sum of squares: 0^2 + 1^2 + 2^2 + 3^2 + 4^2 = 0 + 1 + 4 + 9 + 16 = 30
    """
    res = 0                    # Initialize result to 0
    
    for i in range(5):         # Loop 5 times (i = 0, 1, 2, 3, 4)
        res += i*i             # Add i squared to result
        sleep(1)               # Wait 1 second (simulates slow work)
    
    return res                 # Return the final result

**Let's trace through what happens:**

| Iteration (i) | i*i | res (before) | res (after) |
|---------------|-----|--------------|-------------|
| 0 | 0 | 0 | 0 |
| 1 | 1 | 0 | 1 |
| 2 | 4 | 1 | 5 |
| 3 | 9 | 5 | 14 |
| 4 | 16 | 14 | 30 |

Final result: **30**

In [12]:
# Run the slow calculation
# This will take about 5 seconds (1 second per iteration)
slow_calculation()

30

**Problem:** The function runs silently for 5 seconds with no feedback. In deep learning, training can take hours or days - we want to see progress!

**Solution:** Add a callback that gets called after each iteration.

---

In [68]:
# =====================================================
# A SLOW CALCULATION (VERSION 2 - WITH CALLBACK)
# =====================================================

def slow_calculation(cb=None):
    """
    Same slow calculation, but now accepts an optional callback.
    
    Arguments:
        cb - A callback function (optional, defaults to None)
             If provided, it will be called after each iteration
             The callback receives the current iteration number as argument
    
    Returns:
        The sum of squares (30)
    """
    res = 0                    # Initialize result
    
    for i in range(5):         # Loop 5 times
        res += i*i             # Do the calculation
        sleep(1)               # Wait 1 second
        
        # ---- THE CALLBACK MAGIC ----
        # If a callback was provided (cb is not None), call it!
        # We pass the current iteration number 'i' to the callback
        if cb:                 # Same as: if cb is not None:
            cb(i)              # Call the callback with iteration number
    
    return res

**Understanding the callback pattern:**

```python
if cb: cb(i)
```

This is a common Python idiom:
- `if cb:` checks if `cb` is "truthy" (not None, not empty, etc.)
- If a callback was provided, `cb(i)` calls it with the current iteration

This pattern allows the function to work with or without a callback:
- `slow_calculation()` - works fine, no output during execution
- `slow_calculation(my_function)` - calls `my_function` each iteration

---

Now let's create a callback function to show progress:

In [14]:
# =====================================================
# A CALLBACK FUNCTION FOR PROGRESS REPORTING
# =====================================================

def show_progress(epoch):
    """
    Prints a message showing which epoch (iteration) just finished.
    
    Arguments:
        epoch - The iteration number that just completed
                (we call it 'epoch' because in deep learning,
                 one pass through all training data is called an epoch)
    """
    print(f"Awesome! We've finished epoch {epoch}!")

**About f-strings:** The `f"..."` syntax is called an f-string (formatted string literal). It lets you embed Python expressions inside strings:

```python
name = "Alice"
age = 30
print(f"{name} is {age} years old")  # Output: Alice is 30 years old
```

In [15]:
# =====================================================
# USING THE CALLBACK
# =====================================================

# Now we call slow_calculation and PASS our show_progress function
# Notice: we pass show_progress (the function) not show_progress() (calling it)
#
# What happens:
# 1. slow_calculation starts
# 2. After each iteration, it calls cb(i) which is show_progress(i)
# 3. show_progress prints a message
# 4. This repeats 5 times
# 5. slow_calculation returns 30

slow_calculation(show_progress)

Awesome! We've finished epoch 0!
Awesome! We've finished epoch 1!
Awesome! We've finished epoch 2!
Awesome! We've finished epoch 3!
Awesome! We've finished epoch 4!


30

**What just happened:**

1. We called `slow_calculation(show_progress)`
2. Inside `slow_calculation`, the parameter `cb` now refers to our `show_progress` function
3. Each time through the loop, `cb(i)` is executed, which is the same as `show_progress(i)`
4. Our callback printed a progress message for each iteration
5. Finally, the function returned 30

**This is the power of callbacks:** We customized the behavior of `slow_calculation` without changing its code!

---

### 🎮 Interactive: the Callback Playground

Two sides of the same idea, side by side:

- **Left — the GUI analogy.** A button can't know what should happen when it's clicked; *you* hand it a function. Flip the *register* switch and click the button: with no callback, clicks do nothing; with one, each click travels down the wire into `f` and prints `hi`.
- **Right — `slow_calculation`.** The exact code from above, executed one micro-step at a time. Choose `cb = None`, `show_progress`, or a `lambda` and watch how the *same loop* is silent or chatty depending only on what you passed in.

**Controls:** ⏮ / ⏭ step one micro-step at a time · ▶ Play auto-runs it (🐢–🐇 slider sets the speed) · drag the **scrubber** to any step · and **click the highlighted stages** to jump straight there — the matching **code line lights up** and the purple panel explains that exact moment in plain English.

Try this: play the `None` run, then **click epoch 3** in the `show_progress` run and compare what the `if cb:` line does in each.

In [16]:
# 📊 Interactive visualization — run this cell to display it inline.
# The HTML file lives in the folder `06_foundations_visuals/`, which must sit
# next to this notebook. If your editor blocks the iframe (e.g. some VS Code
# setups), just open the HTML file directly in your browser instead.
from IPython.display import HTML
HTML("""
<iframe src="06_foundations_visuals/callback_playground.html"
        style="width:100%; height:600px; border:1px solid #dde5f2;
               border-radius:12px; box-shadow:0 8px 24px rgba(123,92,214,.12);"
        loading="lazy"></iframe>
<script>addEventListener("message",function(e){if(e.data&&e.data.type==="ce-frame-height"&&e.data.height>50){var fs=document.querySelectorAll("iframe");for(var i=0;i<fs.length;i++){if(fs[i].contentWindow===e.source){fs[i].style.height=e.data.height+"px";break;}}}});</script>
""")

---

## 1.3 Lambdas and Partials

Sometimes we want a simple callback but don't want to define a whole function for it. Python provides two ways to create "lightweight" functions:

1. **Lambda functions** - Anonymous, one-line functions
2. **Partial functions** - Regular functions with some arguments pre-filled

---

### Lambda Functions

A **lambda** is a small anonymous function. "Anonymous" means it doesn't have a name.

**Syntax:**
```python
lambda arguments: expression
```

**Examples:**
```python
# Regular function
def add(x, y):
    return x + y

# Equivalent lambda
add = lambda x, y: x + y

# Both can be called the same way:
add(2, 3)  # Returns 5
```

**Key differences:**
- Lambdas can only contain a single expression (no statements like `if`, loops, etc.)
- Lambdas automatically return the result of the expression
- Lambdas are often used inline where a function is needed briefly

In [17]:
# =====================================================
# LAMBDA EXAMPLES
# =====================================================

# Example 1: A lambda that doubles a number
double = lambda x: x * 2
print(f"double(5) = {double(5)}")  # Output: 10

# Example 2: A lambda that adds two numbers
add = lambda a, b: a + b
print(f"add(3, 4) = {add(3, 4)}")  # Output: 7

# Example 3: A lambda that takes no arguments
greet = lambda: "Hello!"
print(f"greet() = {greet()}")  # Output: Hello!

double(5) = 10
add(3, 4) = 7
greet() = Hello!


In [18]:
# =====================================================
# USING A LAMBDA AS A CALLBACK
# =====================================================

# Instead of defining show_progress, we can use a lambda directly:
#
# lambda o: print(f"Awesome! We've finished epoch {o}!")
#
# Breaking it down:
#   lambda        - keyword to create a lambda function
#   o             - the parameter (will receive the epoch number)
#   :             - separates parameters from the expression
#   print(...)    - the expression to evaluate (prints the message)

slow_calculation(lambda o: print(f"Awesome! We've finished epoch {o}!"))

Awesome! We've finished epoch 0!
Awesome! We've finished epoch 1!
Awesome! We've finished epoch 2!
Awesome! We've finished epoch 3!
Awesome! We've finished epoch 4!


30

**When to use lambdas:**
- For simple, one-line functions
- When you need a function only in one place
- When passing a simple function as an argument

**When NOT to use lambdas:**
- For complex logic (use regular functions)
- When you need to reuse the function (give it a name)
- When readability suffers

---

### The Problem: Callbacks with Multiple Parameters

What if our callback needs more information? Let's say we want to customize the exclamation:

In [19]:
# =====================================================
# A CALLBACK WITH TWO PARAMETERS
# =====================================================

def show_progress(exclamation, epoch):
    """
    Prints progress with a customizable exclamation.
    
    Arguments:
        exclamation - A string like "Awesome" or "Great" or "OK I guess"
        epoch       - The current iteration number
    """
    print(f"{exclamation}! We've finished epoch {epoch}!")

**Problem:** Our `slow_calculation` calls `cb(i)` - it only passes ONE argument (the epoch number). But now `show_progress` needs TWO arguments (exclamation AND epoch).

How do we solve this?

---

**Solution 1: Use a Lambda as a Wrapper**

In [20]:
# =====================================================
# USING LAMBDA TO "FILL IN" AN ARGUMENT
# =====================================================

# The lambda takes one argument (o = epoch number from slow_calculation)
# and calls show_progress with TWO arguments:
#   - "OK I guess" (the exclamation we want)
#   - o (the epoch number passed to the lambda)
#
# This is called "wrapping" - the lambda wraps show_progress

slow_calculation(lambda o: show_progress("OK I guess", o))

OK I guess! We've finished epoch 0!
OK I guess! We've finished epoch 1!
OK I guess! We've finished epoch 2!
OK I guess! We've finished epoch 3!
OK I guess! We've finished epoch 4!


30

**How it works:**

```
slow_calculation calls:  cb(i)  where i is 0, 1, 2, 3, 4
        |
        v
lambda receives:         o      (o gets the value of i)
        |
        v
lambda calls:            show_progress("OK I guess", o)
        |
        v
show_progress receives:  exclamation="OK I guess", epoch=o
```

---

**Solution 2: Create a Function that Returns a Function (Closure)**

This is a more advanced pattern but very useful.

In [21]:
# =====================================================
# A FUNCTION THAT CREATES CALLBACK FUNCTIONS
# =====================================================

def make_show_progress(exclamation):
    """
    Creates and returns a callback function with the exclamation "baked in".
    
    This is called a "closure" - the inner function "closes over" 
    (remembers) the exclamation variable from the outer function.
    
    Arguments:
        exclamation - The exclamation to use in messages
    
    Returns:
        A function that takes one argument (epoch) and prints a message
    """
    
    # Define an inner function
    def _inner(epoch):
        # This function can access 'exclamation' from the outer function!
        print(f"{exclamation}! We've finished epoch {epoch}!")
    
    # Return the inner function (not call it - no parentheses)
    return _inner

In [22]:
# =====================================================
# USING THE CLOSURE
# =====================================================

# make_show_progress("Nice!") returns a function
# That function remembers exclamation="Nice!"
# We pass this function as the callback

slow_calculation(make_show_progress("Nice!"))

Nice!! We've finished epoch 0!
Nice!! We've finished epoch 1!
Nice!! We've finished epoch 2!
Nice!! We've finished epoch 3!
Nice!! We've finished epoch 4!


30

**Step by step:**

1. `make_show_progress("Nice!")` is called
2. Inside, `exclamation = "Nice!"`
3. The `_inner` function is created - it remembers that `exclamation = "Nice!"`
4. `_inner` is returned
5. `slow_calculation` receives `_inner` as its callback
6. When `slow_calculation` calls `cb(i)`, it's calling `_inner(i)`
7. `_inner` uses the remembered `exclamation` value

---

**Solution 3: Use `functools.partial` (The Recommended Way)**

`partial` is a Python utility that does exactly what we need: it creates a new function with some arguments pre-filled.

In [23]:
# Import partial from functools (a standard Python module)
from functools import partial

In [24]:
# =====================================================
# USING PARTIAL TO PRE-FILL ARGUMENTS
# =====================================================

# partial(function, arg1, arg2, ...) creates a new function
# with the first arguments pre-filled
#
# partial(show_progress, "OK I guess") creates a function that:
#   - Already has exclamation="OK I guess"
#   - Only needs the epoch argument
#
# This is equivalent to:
#   lambda epoch: show_progress("OK I guess", epoch)

slow_calculation(partial(show_progress, "OK I guess"))

OK I guess! We've finished epoch 0!
OK I guess! We've finished epoch 1!
OK I guess! We've finished epoch 2!
OK I guess! We've finished epoch 3!
OK I guess! We've finished epoch 4!


30

In [25]:
# =====================================================
# UNDERSTANDING PARTIAL BETTER
# =====================================================

# Let's save the partial function to a variable to understand it better
f2 = partial(show_progress, "OK I guess")

# f2 is now a function that only needs the 'epoch' argument
print("Type of f2:", type(f2))

# Let's call it directly:
print("Calling f2(42):")
f2(42)  # Same as: show_progress("OK I guess", 42)

Type of f2: <class 'functools.partial'>
Calling f2(42):
OK I guess! We've finished epoch 42!


**Summary of the three approaches:**

| Approach | Code | Best for |
|----------|------|----------|
| Lambda wrapper | `lambda o: show_progress("OK", o)` | Simple, one-time use |
| Closure | `make_show_progress("OK")` | When you need custom logic in the factory |
| Partial | `partial(show_progress, "OK")` | Most cases - clean and Pythonic |

---

### 🎮 Interactive: the Argument-Filling Machine

`show_progress(exclamation, epoch)` takes **two** arguments, but the loop only ever calls `cb(i)` with **one**. All three tabs — **lambda**, **closure**, **partial** — build the same *wrapper machine* that already contains the exclamation.

Notice **step 1** especially: the wrapper is built **once**, *before any epoch runs* — the purple value is baked in at that moment and never changes. From then on, each epoch only supplies the orange `i`. Type your own exclamation and switch tabs: the machine in the middle stays identical.

**Controls:** ⏮ / ⏭ step one micro-step at a time · ▶ Play auto-runs it (🐢–🐇 slider sets the speed) · drag the **scrubber** to any step · and **click the highlighted stages** to jump straight there — the matching **code line lights up** and the purple panel explains that exact moment in plain English. The `build` pill jumps to the wrapper's creation; the `epoch n` pills jump into the loop.

In [26]:
# 📊 Interactive visualization — run this cell to display it inline.
# The HTML file lives in the folder `06_foundations_visuals/`, which must sit
# next to this notebook. If your editor blocks the iframe (e.g. some VS Code
# setups), just open the HTML file directly in your browser instead.
from IPython.display import HTML
HTML("""
<iframe src="06_foundations_visuals/lambda_partial_machine.html"
        style="width:100%; height:600px; border:1px solid #dde5f2;
               border-radius:12px; box-shadow:0 8px 24px rgba(123,92,214,.12);"
        loading="lazy"></iframe>
<script>addEventListener("message",function(e){if(e.data&&e.data.type==="ce-frame-height"&&e.data.height>50){var fs=document.querySelectorAll("iframe");for(var i=0;i<fs.length;i++){if(fs[i].contentWindow===e.source){fs[i].style.height=e.data.height+"px";break;}}}});</script>
""")

---

## 1.4 Callbacks as Callable Classes

So far, our callbacks have been functions. But Python allows us to make **objects** that can be called like functions. These are called "callable" objects.

**Why use a class instead of a function?**
- Classes can store state (data that persists between calls)
- Classes can be configured when created
- Classes can have multiple methods, not just one

To make a class callable, we define the special method `__call__`.

---

In [27]:
# =====================================================
# A CALLABLE CLASS
# =====================================================

class ProgressShowingCallback():
    """
    A callback class that shows progress messages.
    
    This class can be CALLED like a function because it has __call__.
    
    When you create an instance: cb = ProgressShowingCallback("Wow")
    You can then call it: cb(5) - this runs __call__(5)
    """
    
    def __init__(self, exclamation="Awesome"):
        """
        Initialize the callback with an exclamation.
        
        Arguments:
            exclamation - The exclamation to use (default: "Awesome")
        
        This is called when we create an instance:
            cb = ProgressShowingCallback("Wow")
        """
        # Store the exclamation as an instance variable
        # 'self' refers to the object being created
        self.exclamation = exclamation
    
    def __call__(self, epoch):
        """
        This method is called when the object is used like a function.
        
        Arguments:
            epoch - The current iteration number
        
        If we have: cb = ProgressShowingCallback("Wow")
        Then: cb(5) is equivalent to cb.__call__(5)
        """
        print(f"{self.exclamation}! We've finished epoch {epoch}!")

In [69]:
# =====================================================
# CREATING AND USING A CALLABLE OBJECT
# =====================================================

# Step 1: Create an instance of the class
# This calls __init__ with exclamation="Just super"
cb = ProgressShowingCallback("Just super")

# cb is now an object that:
# - Has cb.exclamation = "Just super"
# - Can be called like a function: cb(epoch)

In [70]:
cb(0)

Just super! We've finished epoch 0!


In [71]:
# import inspect
# print(inspect.getsource(slow_calculation))

In [72]:
# Step 2: Use the callable object as a callback
# slow_calculation will call cb(i) for each iteration
# This is the same as calling cb.__call__(i)

slow_calculation(cb)

Just super! We've finished epoch 0!
Just super! We've finished epoch 1!
Just super! We've finished epoch 2!
Just super! We've finished epoch 3!
Just super! We've finished epoch 4!


30

**Why this is powerful:**

1. The callback is configured at creation time (`"Just super"`)
2. The object remembers its configuration (`self.exclamation`)
3. We can call it just like a function
4. We could add more state - for example, counting how many times it was called

**Example: A callback that counts calls**

```python
class CountingCallback():
    def __init__(self):
        self.count = 0  # Initialize counter
    
    def __call__(self, epoch):
        self.count += 1  # Increment counter
        print(f"Called {self.count} times, currently at epoch {epoch}")
```

---

---

## 1.5 Multiple Callback Functions; `*args` and `**kwargs`

Real callbacks often need multiple methods (e.g., `before_training`, `after_epoch`, `after_training`). Before we tackle that, let's learn about Python's flexible argument handling.

---

### Understanding `*args` and `**kwargs`

Python has special syntax for handling variable numbers of arguments:

- `*args` - Captures any number of **positional** arguments as a tuple
- `**kwargs` - Captures any number of **keyword** arguments as a dictionary

("args" is short for "arguments", "kwargs" is short for "keyword arguments")

---

**What are positional vs keyword arguments?**

```python
def greet(name, greeting):
    print(f"{greeting}, {name}!")

# Positional arguments - matched by position
greet("Alice", "Hello")  # name="Alice", greeting="Hello"

# Keyword arguments - matched by name
greet(name="Alice", greeting="Hello")
greet(greeting="Hi", name="Bob")  # Order doesn't matter with keywords
```

In [31]:
# =====================================================
# DEMONSTRATION OF *args AND **kwargs
# =====================================================

def f(*a, **b):
    """
    A function that accepts any number of arguments.
    
    *a   - Collects all positional arguments into a tuple called 'a'
    **b  - Collects all keyword arguments into a dictionary called 'b'
    
    Note: 'a' and 'b' are just names - the * and ** are what matter.
    Convention is to use 'args' and 'kwargs' but any name works.
    """
    print(f"args: {a}; kwargs: {b}")

In [32]:
# =====================================================
# CALLING WITH VARIOUS ARGUMENTS
# =====================================================

# f(3, 'a', thing1="hello")
#
# Breaking down the call:
#   3         - positional argument (goes into *a)
#   'a'       - positional argument (goes into *a)
#   thing1="hello"  - keyword argument (goes into **b)
#
# Inside f:
#   a = (3, 'a')              - a tuple of positional args
#   b = {'thing1': 'hello'}   - a dict of keyword args

f(3, 'a', thing1="hello")

args: (3, 'a'); kwargs: {'thing1': 'hello'}


**More examples:**

In [33]:
# No arguments
print("f() with no args:")
f()

print()

# Only positional
print("f(1, 2, 3):")
f(1, 2, 3)

print()

# Only keyword
print("f(x=1, y=2):")
f(x=1, y=2)

print()

# Mixed
print("f(1, 2, a='hello', b='world'):")
f(1, 2, a='hello', b='world')

f() with no args:
args: (); kwargs: {}

f(1, 2, 3):
args: (1, 2, 3); kwargs: {}

f(x=1, y=2):
args: (); kwargs: {'x': 1, 'y': 2}

f(1, 2, a='hello', b='world'):
args: (1, 2); kwargs: {'a': 'hello', 'b': 'world'}


---

### Unpacking with `*` and `**`

The `*` and `**` operators can also be used to **unpack** (spread out) sequences and dictionaries when calling functions.

In [34]:
# =====================================================
# UNPACKING ARGUMENTS
# =====================================================

def g(a, b, c=0):
    """
    A function with specific parameters:
        a - first required parameter
        b - second required parameter  
        c - optional parameter with default value 0
    """
    print(a, b, c)

In [35]:
# =====================================================
# USING * AND ** TO UNPACK WHEN CALLING
# =====================================================

# We have a list of positional arguments and a dict of keyword arguments
args = [1, 2]        # This will become a=1, b=2
kwargs = {'c': 3}    # This will become c=3

# *args unpacks the list: [1, 2] becomes 1, 2
# **kwargs unpacks the dict: {'c': 3} becomes c=3
#
# So g(*args, **kwargs) is equivalent to g(1, 2, c=3)

g(*args, **kwargs)

1 2 3


**This is incredibly useful for:**
- Passing arguments through from one function to another
- Building arguments dynamically
- Creating flexible APIs

---

### Why This Matters for Callbacks

Using `*args` and `**kwargs` in callbacks allows:
1. The main function to pass extra information (like current values)
2. Callbacks to ignore information they don't need
3. Future compatibility - you can add new parameters without breaking old callbacks

---

### 🎮 Interactive: the `*args` / `**kwargs` Sorter

- **Card A — packing (in a `def`).** Build any call to `f(*a, **b)` with the chip builder, then step through Python sorting the arguments **one at a time**: each step states the rule it applied (*no `name=` → tuple; `name=value` → dict*). Click any **chip** or **step pill** to jump to the exact moment that argument is sorted (✕ removes a chip).
- **Card B — unpacking (in a call).** The same stars in reverse: `g(*args, **kwargs)` spreads containers into named slots. **Click a parameter slot** (`a`, `b`, `c`) to jump to the moment it gets filled.

**Controls:** ⏮ / ⏭ step one micro-step at a time · ▶ Play auto-runs it (🐢–🐇 slider sets the speed) · drag the **scrubber** to any step · and **click the highlighted stages** to jump straight there — the matching **code line lights up** and the purple panel explains that exact moment in plain English. The 🐢–🐇 slider at the top controls both cards.

In [36]:
# 📊 Interactive visualization — run this cell to display it inline.
# The HTML file lives in the folder `06_foundations_visuals/`, which must sit
# next to this notebook. If your editor blocks the iframe (e.g. some VS Code
# setups), just open the HTML file directly in your browser instead.
from IPython.display import HTML
HTML("""
<iframe src="06_foundations_visuals/args_kwargs_sorter.html"
        style="width:100%; height:600px; border:1px solid #dde5f2;
               border-radius:12px; box-shadow:0 8px 24px rgba(123,92,214,.12);"
        loading="lazy"></iframe>
<script>addEventListener("message",function(e){if(e.data&&e.data.type==="ce-frame-height"&&e.data.height>50){var fs=document.querySelectorAll("iframe");for(var i=0;i<fs.length;i++){if(fs[i].contentWindow===e.source){fs[i].style.height=e.data.height+"px";break;}}}});</script>
""")

### Callbacks with Multiple Methods

Now let's create a more sophisticated callback system with multiple "hooks" - points where callbacks can be triggered.

In [37]:
# =====================================================
# SLOW CALCULATION WITH MULTIPLE CALLBACK POINTS
# =====================================================

def slow_calculation(cb=None):
    """
    A slow calculation with TWO callback points:
    1. before_calc - called BEFORE each calculation
    2. after_calc  - called AFTER each calculation
    
    The callback object (cb) should have these methods if it wants
    to respond to these events.
    
    Arguments:
        cb - A callback object with before_calc and/or after_calc methods
    """
    res = 0
    
    for i in range(5):
        # --- BEFORE CALCULATION HOOK ---
        # Call cb.before_calc(i) if cb exists
        # We pass i as a positional argument
        if cb: 
            cb.before_calc(i)
        
        # The actual calculation
        res += i*i
        sleep(1)
        
        # --- AFTER CALCULATION HOOK ---
        # Call cb.after_calc with epoch AND current value
        # We pass i as positional, and val as keyword argument
        if cb: 
            cb.after_calc(i, val=res)
    
    return res

In [38]:
# =====================================================
# A SIMPLE CALLBACK USING *args AND **kwargs
# =====================================================

class PrintStepCallback():
    """
    A callback that just prints a message before and after each step.
    
    It uses *args and **kwargs to ACCEPT any arguments,
    even though it doesn't use them. This is a common pattern.
    
    Why do this?
    - The main function might pass different info to different callbacks
    - This callback doesn't care about the specifics
    - Using *args, **kwargs means it won't break if arguments change
    """
    
    def before_calc(self, *args, **kwargs):
        # We accept any arguments but ignore them
        print(f"About to start")
    
    def after_calc(self, *args, **kwargs):
        # We accept any arguments but ignore them
        print(f"Done step")

In [39]:
# Run with PrintStepCallback
slow_calculation(PrintStepCallback())

About to start
Done step
About to start
Done step
About to start
Done step
About to start
Done step
About to start
Done step


30

**Notice:** The callback doesn't use any information from the main function - it just prints generic messages. Let's make a smarter callback:

In [40]:
# =====================================================
# A CALLBACK THAT USES SOME ARGUMENTS
# =====================================================

class PrintStatusCallback():
    """
    A callback that prints the actual epoch number and value.
    
    It uses NAMED parameters for what it needs, and **kwargs
    to capture (and ignore) anything else.
    """
    
    def __init__(self): 
        pass  # Nothing to initialize
    
    def before_calc(self, epoch, **kwargs):
        """
        Called before each calculation.
        
        Arguments:
            epoch    - The current iteration (we use this!)
            **kwargs - Any other keyword args (we ignore these)
        """
        print(f"About to start: {epoch}")
    
    def after_calc(self, epoch, val, **kwargs):
        """
        Called after each calculation.
        
        Arguments:
            epoch    - The current iteration (we use this!)
            val      - The current calculated value (we use this!)
            **kwargs - Any other keyword args (we ignore these)
        """
        print(f"After {epoch}: {val}")

In [41]:
# Run with PrintStatusCallback - now we see actual values!
slow_calculation(PrintStatusCallback())

About to start: 0
After 0: 0
About to start: 1
After 1: 1
About to start: 2
After 2: 5
About to start: 3
After 3: 14
About to start: 4
After 4: 30


30

**Key insight:** By using `**kwargs`, the callback can:
- Take only the arguments it needs by name (`epoch`, `val`)
- Ignore any extra arguments without crashing
- Work even if the main function adds new parameters later

---

---

## 1.6 Modifying Behavior with Callbacks

So far, callbacks just observed and reported. But callbacks can also **modify behavior** - for example:
- Stop training early
- Skip certain iterations
- Modify values

Let's implement a callback that can stop the calculation early.

In [42]:
# =====================================================
# SLOW CALCULATION THAT CAN BE STOPPED EARLY
# =====================================================

def slow_calculation(cb=None):
    """
    A slow calculation where:
    1. Callbacks are optional (we check if they exist)
    2. The after_calc callback can return True to stop early
    
    New features:
    - Uses hasattr() to check if the callback has each method
    - Checks the return value of after_calc
    - Breaks out of the loop if callback returns True
    """
    res = 0
    
    for i in range(5):
        # --- BEFORE CALCULATION ---
        # Check if cb exists AND has a 'before_calc' method
        # hasattr(object, name) returns True if object has attribute 'name'
        if cb and hasattr(cb, 'before_calc'): 
            cb.before_calc(i)
        
        res += i*i
        sleep(1)
        
        # --- AFTER CALCULATION ---
        # Check if cb exists AND has 'after_calc'
        if cb and hasattr(cb, 'after_calc'):
            # The callback can return True to signal "stop early"
            if cb.after_calc(i, res):   # If returns True...
                print("stopping early")
                break                    # ...break out of the loop
    
    return res

**New concepts used:**

1. **`hasattr(obj, name)`** - Returns `True` if `obj` has an attribute (variable or method) called `name`
   ```python
   hasattr(cb, 'before_calc')  # Does cb have a before_calc method?
   ```

2. **Checking return value of callback** - If `after_calc` returns `True`, we stop early
   ```python
   if cb.after_calc(i, res):  # If returns truthy value
       break                   # Stop the loop
   ```

3. **`break` statement** - Exits the current loop immediately

In [43]:
# =====================================================
# A CALLBACK THAT STOPS EARLY
# =====================================================

class PrintAfterCallback():
    """
    A callback that:
    1. Prints the value after each step
    2. Returns True (to stop) if value exceeds 10
    
    Note: This callback only has after_calc, not before_calc.
    That's OK - the main function checks with hasattr.
    """
    
    def after_calc(self, epoch, val):
        """
        Called after each calculation.
        
        Arguments:
            epoch - Current iteration number
            val   - Current calculated value
        
        Returns:
            True if val > 10 (signals to stop early)
            None otherwise (implicitly, same as returning nothing)
        """
        print(f"After {epoch}: {val}")
        
        # If value is greater than 10, return True to stop
        if val > 10: 
            return True

In [44]:
# Run with PrintAfterCallback - it should stop early!
slow_calculation(PrintAfterCallback())

After 0: 0
After 1: 1
After 2: 5
After 3: 14
stopping early


14

**What happened:**

| Epoch | res after | val > 10? | Action |
|-------|-----------|-----------|--------|
| 0 | 0 | No | Continue |
| 1 | 1 | No | Continue |
| 2 | 5 | No | Continue |
| 3 | 14 | **Yes** | **Return True, stop!** |
| 4 | (never reached) | | |

The final value is 14, not 30, because we stopped after epoch 3.

---

# Callback Terminology

`slow_calculation` is **not a class** — it's a **function**. There is no class named `slow_calculation` anywhere in the notebook.

## Naming each piece correctly

**`slow_calculation`** is the **function** that does the work (the sum of squares). In callback terminology it's often called the **caller** — the thing that *runs* the callbacks. It decides *when* to invoke them (after each epoch, before each calc, etc.).

**`PrintAfterCallback()`** creates an **instance** (an object) of the `PrintAfterCallback` **class**. That object *is* the **callback** — it's the thing being passed in and "called back" by `slow_calculation`.

## So in `slow_calculation(PrintAfterCallback())`

- `slow_calculation` → the **function / caller**
- `PrintAfterCallback` → a **class**
- `PrintAfterCallback()` → an **instance** of that class, and *this* is the **callback**

## The pattern

The caller (`slow_calculation`) accepts a callback object and "calls it back" at defined points (`cb.after_calc(...)`). The callback object decides what to *do* at those points.

This is the **inversion-of-control** idea:

- `slow_calculation` controls **when**
- the callback controls **what**

### 🎮 Interactive: Hooks, `hasattr` Gates & Early Stopping

Each epoch flows through three nodes — `before_calc` → `res += i*i` → `after_calc` — and each hook sits behind a **gate**: `if cb and hasattr(cb, '...')`. Both code panels are shown (the loop *and* your callback) and the active lines light up in each as you step.

**Every node, gate and epoch label is clickable** — jump to the before-gate of epoch 2, or straight to the moment `after_calc` returns `True`. Clicking a grayed-out epoch after an early stop explains *why it never ran*.

**Controls:** ⏮ / ⏭ step one micro-step at a time · ▶ Play auto-runs it (🐢–🐇 slider sets the speed) · drag the **scrubber** to any step · and **click the highlighted stages** to jump straight there — the matching **code line lights up** and the purple panel explains that exact moment in plain English.

Predict first: `res` goes 0, 1, 5, 14, 30 — with the early-stop rule `val > 10`, which epoch is the last to run? Untick `before_calc` afterwards: nothing crashes. Why?

In [45]:
# 📊 Interactive visualization — run this cell to display it inline.
# The HTML file lives in the folder `06_foundations_visuals/`, which must sit
# next to this notebook. If your editor blocks the iframe (e.g. some VS Code
# setups), just open the HTML file directly in your browser instead.
from IPython.display import HTML
HTML("""
<iframe src="06_foundations_visuals/callback_hooks_timeline.html"
        style="width:100%; height:600px; border:1px solid #dde5f2;
               border-radius:12px; box-shadow:0 8px 24px rgba(123,92,214,.12);"
        loading="lazy"></iframe>
<script>addEventListener("message",function(e){if(e.data&&e.data.type==="ce-frame-height"&&e.data.height>50){var fs=document.querySelectorAll("iframe");for(var i=0;i<fs.length;i++){if(fs[i].contentWindow===e.source){fs[i].style.height=e.data.height+"px";break;}}}});</script>
""")

### A More Sophisticated Callback System

Now let's build a proper calculator class where callbacks can even **modify the state**.

In [46]:
# =====================================================
# A CALCULATOR CLASS WITH CALLBACK SYSTEM
# =====================================================

class SlowCalculator():
    """
    A calculator that performs a slow calculation with callback support.
    
    Key features:
    1. Stores state (self.res) that callbacks can access and modify
    2. Has a general callback() method that handles calling the right method
    3. Passes itself to callbacks so they can access/modify state
    """
    
    def __init__(self, cb=None):
        """
        Initialize the calculator.
        
        Arguments:
            cb - Optional callback object
        """
        self.cb = cb        # Store the callback
        self.res = 0        # Initialize the result
    
    def callback(self, cb_name, *args):
        """
        A helper method to call callbacks safely.
        
        Arguments:
            cb_name - The name of the callback method (e.g., 'before_calc')
            *args   - Arguments to pass to the callback
        
        Returns:
            Whatever the callback returns, or None if no callback
        
        How it works:
        1. If no callback object, return None
        2. Try to get the method with the given name
        3. If method exists, call it with self (the calculator) and args
        """
        # If no callback object was provided, do nothing
        if not self.cb: 
            return
        
        # Try to get the method named cb_name from the callback object
        # getattr(obj, name, default) returns obj.name or default if not found
        cb = getattr(self.cb, cb_name, None)
        
        # If the method exists, call it
        # We pass 'self' (the calculator) so callback can access self.res
        if cb: 
            return cb(self, *args)
    
    def calc(self):
        """
        Perform the slow calculation.
        
        Uses self.callback() to trigger callbacks at appropriate times.
        Callbacks can access and modify self.res!
        """
        for i in range(5):
            # Call 'before_calc' callback with current epoch
            self.callback('before_calc', i)
            
            # Do the actual calculation
            self.res += i*i
            sleep(1)
            
            # Call 'after_calc' callback
            # If it returns True, stop early
            if self.callback('after_calc', i):
                print("stopping early")
                break

**Key design patterns in this class:**

1. **State is stored in the object** (`self.res`) - callbacks can access and modify it

2. **The `callback` helper method** - centralizes the logic for calling callbacks:
   - Checks if callback exists
   - Gets the right method by name
   - Passes the calculator object (`self`) to the callback

3. **`getattr(obj, name, default)`** - A powerful Python function:
   ```python
   getattr(self.cb, 'before_calc', None)
   # Returns self.cb.before_calc if it exists
   # Returns None if it doesn't exist
   ```

---

In [47]:
# =====================================================
# A CALLBACK THAT MODIFIES STATE
# =====================================================

class ModifyingCallback():
    """
    A callback that can:
    1. Print the current state
    2. Stop early if value exceeds 10
    3. MODIFY the result if it's less than 3 (doubles it!)
    
    This demonstrates how callbacks can actually change behavior!
    """
    
    def after_calc(self, calc, epoch):
        """
        Called after each calculation.
        
        Arguments:
            calc  - The SlowCalculator object (we can access calc.res!)
            epoch - The current iteration number
        
        Returns:
            True to stop early, None otherwise
        """
        # Print current state
        print(f"After {epoch}: {calc.res}")
        
        # Check if we should stop
        if calc.res > 10: 
            return True
        
        # If result is less than 3, DOUBLE IT!
        # This modifies the calculator's state!
        if calc.res < 3: 
            calc.res = calc.res * 2

In [48]:
# =====================================================
# USING THE MODIFYING CALLBACK
# =====================================================

# Create a calculator with our modifying callback
calculator = SlowCalculator(ModifyingCallback())

In [49]:
# Run the calculation and see the final result
calculator.calc()
calculator.res

After 0: 0
After 1: 1
After 2: 6
After 3: 15
stopping early


15

**What happened (step by step):**

| Epoch | res after calc | res < 3? | Action | res after callback |
|-------|----------------|----------|--------|--------------------|
| 0 | 0 | Yes | Double it! | 0 (0*2=0) |
| 1 | 1 | Yes | Double it! | 2 (1*2=2) |
| 2 | 6 | No | Keep it | 6 |
| 3 | 15 | No | res > 10, STOP! | 15 |

Wait, let me trace this more carefully:

- **Epoch 0:** res = 0 + 0*0 = 0, then callback doubles: 0*2 = 0
- **Epoch 1:** res = 0 + 1*1 = 1, then callback doubles: 1*2 = 2
- **Epoch 2:** res = 2 + 2*2 = 6, res >= 3 so no doubling
- **Epoch 3:** res = 6 + 3*3 = 15, res > 10 so STOP

Final result: **15**

**The callback actually changed the computation!** Without the callback, the result would be 30. With the callback modifying intermediate values and stopping early, we get 15.

---

### 🎮 Interactive: a Callback That *Changes* the Calculation

`SlowCalculator` passes **itself** (`self`) to its callback, so `ModifyingCallback` can *read and write* `calc.res`. Each epoch has three micro-steps: the loop **writes** (orange bar), the callback **reads & prints**, then the callback **decides** — double it (`res < 3`), stop everything (`res > 10`), or leave it (purple bar).

**Click any epoch's bars** to jump straight to it. When the purple bar is taller than the orange one, you're literally seeing shared state being mutated from outside the loop — the same mechanism behind fastai's LR schedulers and early stopping.

**Controls:** ⏮ / ⏭ step one micro-step at a time · ▶ Play auto-runs it (🐢–🐇 slider sets the speed) · drag the **scrubber** to any step · and **click the highlighted stages** to jump straight there — the matching **code line lights up** and the purple panel explains that exact moment in plain English. Toggle the callback off, replay, and compare final results (15 vs 30).

In [50]:
# 📊 Interactive visualization — run this cell to display it inline.
# The HTML file lives in the folder `06_foundations_visuals/`, which must sit
# next to this notebook. If your editor blocks the iframe (e.g. some VS Code
# setups), just open the HTML file directly in your browser instead.
from IPython.display import HTML
HTML("""
<iframe src="06_foundations_visuals/state_modifying_calculator.html"
        style="width:100%; height:600px; border:1px solid #dde5f2;
               border-radius:12px; box-shadow:0 8px 24px rgba(123,92,214,.12);"
        loading="lazy"></iframe>
<script>addEventListener("message",function(e){if(e.data&&e.data.type==="ce-frame-height"&&e.data.height>50){var fs=document.querySelectorAll("iframe");for(var i=0;i<fs.length;i++){if(fs[i].contentWindow===e.source){fs[i].style.height=e.data.height+"px";break;}}}});</script>
""")

---

# Part 2: `__dunder__` Methods (Magic Methods)

---

## What are Dunder Methods?

In Python, methods with names that start and end with double underscores (like `__init__`, `__add__`, `__call__`) are called "dunder" methods (short for "double underscore"). They're also called:
- **Magic methods** - because Python calls them automatically in certain situations
- **Special methods** - because they have special meaning to the Python interpreter

**How they work:**
- Python defines specific situations when these methods are called
- By implementing them, you can customize how your objects behave
- They enable operator overloading (customizing what `+`, `-`, `[]`, etc. do)

**Examples:**

| When Python sees... | It calls... |
|---------------------|-------------|
| `obj = MyClass()` | `MyClass.__init__(obj)` |
| `a + b` | `a.__add__(b)` |
| `len(obj)` | `obj.__len__()` |
| `obj[key]` | `obj.__getitem__(key)` |
| `str(obj)` or `print(obj)` | `obj.__str__()` |
| Displaying in Jupyter | `obj.__repr__()` |
| `obj(args)` | `obj.__call__(args)` |

---

In [51]:
# =====================================================
# EXAMPLE: CUSTOM CLASS WITH DUNDER METHODS
# =====================================================

class SloppyAdder():
    """
    A class that stores a number and has custom addition behavior.
    
    It's "sloppy" because adding two SloppyAdders adds a small error (0.01)!
    This is just for demonstration.
    
    Demonstrates:
    - __init__: Called when creating an object
    - __add__:  Called when using + operator
    - __repr__: Called when displaying the object
    """
    
    def __init__(self, o):
        """
        Initialize with a value.
        
        Arguments:
            o - The value to store
        
        Called when you write: SloppyAdder(5)
        """
        self.o = o  # Store the value
    
    def __add__(self, b):
        """
        Define what + means for SloppyAdder objects.
        
        Arguments:
            b - Another SloppyAdder to add
        
        Returns:
            A new SloppyAdder with the sum (plus a small error)
        
        Called when you write: a + b (where a is a SloppyAdder)
        """
        # Add the values plus a small "sloppy" error
        return SloppyAdder(self.o + b.o + 0.01)
    
    def __repr__(self):
        """
        Define how to display this object.
        
        Returns:
            A string representation of the object
        
        Called when:
        - Displaying in Jupyter
        - Using repr(obj)
        - Printing in lists/dicts
        """
        return str(self.o)  # Just show the value

In [52]:
# =====================================================
# USING THE SLOPPY ADDER
# =====================================================

# Create two SloppyAdders
a = SloppyAdder(1)  # Calls __init__(1), stores self.o = 1
b = SloppyAdder(2)  # Calls __init__(2), stores self.o = 2

# Add them using +
# This calls a.__add__(b)
# Which returns SloppyAdder(1 + 2 + 0.01) = SloppyAdder(3.01)
result = a + b

# Display the result
# This calls result.__repr__()
# Which returns "3.01"
result

3.01

**Notice:** We can use `+` with our custom class just like with numbers! This is the power of dunder methods - they let you make your objects behave like built-in types.

---

### Important Dunder Methods to Know

Here's a reference of commonly used dunder methods:

| Method | When it's called | Example |
|--------|------------------|--------|
| `__init__` | Creating an object | `obj = MyClass()` |
| `__new__` | Before `__init__`, creates the object | (Advanced) |
| `__del__` | When object is deleted/garbage collected | `del obj` |
| `__repr__` | String representation for developers | `repr(obj)`, Jupyter display |
| `__str__` | String representation for users | `str(obj)`, `print(obj)` |
| `__len__` | Getting length | `len(obj)` |
| `__getitem__` | Indexing with [] | `obj[key]` |
| `__setitem__` | Setting with [] | `obj[key] = value` |
| `__getattr__` | Accessing missing attributes | `obj.missing_attr` |
| `__setattr__` | Setting any attribute | `obj.attr = value` |
| `__call__` | Calling like a function | `obj(args)` |
| `__enter__`, `__exit__` | Context managers | `with obj as x:` |

For more, see the [Python Data Model documentation](https://docs.python.org/3/reference/datamodel.html).

---

### 🎮 Interactive: Dunder Dispatch

- **Card A — the translation dictionary.** Click any expression (`a + b`, `len(a)`, `a[3]`, `a(5)`, …) to see the `__dunder__` call Python actually makes, plus why it matters for PyTorch/fastai.
- **Card B — one `a + b`, frame by frame.** A six-stage pipeline: *you write `a + b` → Python rewrites it to `a.__add__(b)` → the body computes → a **new** SloppyAdder is born via `__init__` → Jupyter calls `__repr__` → you see the number.* **Click any stage to jump to it** — its code line lights up on the left. Drag the `a.o` / `b.o` sliders and every stage recomputes live.

**Controls:** ⏮ / ⏭ step one micro-step at a time · ▶ Play auto-runs it (🐢–🐇 slider sets the speed) · drag the **scrubber** to any step · and **click the highlighted stages** to jump straight there — the matching **code line lights up** and the purple panel explains that exact moment in plain English.

In [53]:
# 📊 Interactive visualization — run this cell to display it inline.
# The HTML file lives in the folder `06_foundations_visuals/`, which must sit
# next to this notebook. If your editor blocks the iframe (e.g. some VS Code
# setups), just open the HTML file directly in your browser instead.
from IPython.display import HTML
HTML("""
<iframe src="06_foundations_visuals/dunder_dispatch.html"
        style="width:100%; height:600px; border:1px solid #dde5f2;
               border-radius:12px; box-shadow:0 8px 24px rgba(123,92,214,.12);"
        loading="lazy"></iframe>
<script>addEventListener("message",function(e){if(e.data&&e.data.type==="ce-frame-height"&&e.data.height>50){var fs=document.querySelectorAll("iframe");for(var i=0;i<fs.length;i++){if(fs[i].contentWindow===e.source){fs[i].style.height=e.data.height+"px";break;}}}});</script>
""")

---

## 2.1 `__getattr__` and `getattr`

These are related but different:

- **`getattr(obj, name)`** - A built-in **function** to get an attribute by name
- **`__getattr__`** - A **method** you define to handle attribute access for missing attributes

---

### The `getattr` Function

`getattr(object, name)` is equivalent to `object.name`, but lets you specify the attribute name as a string. This is useful when the attribute name is stored in a variable or computed dynamically.

In [54]:
# =====================================================
# SIMPLE CLASS FOR DEMONSTRATION
# =====================================================

class A:
    """
    A simple class with two class attributes.
    
    Class attributes are shared by all instances.
    (As opposed to instance attributes set in __init__)
    """
    a = 1  # Class attribute
    b = 2  # Class attribute

In [55]:
# Create an instance of A
a = A()

In [56]:
# =====================================================
# TWO WAYS TO ACCESS ATTRIBUTES
# =====================================================

# Method 1: Dot notation (the usual way)
print(f"a.b = {a.b}")

# Method 2: getattr function
print(f"getattr(a, 'b') = {getattr(a, 'b')}")

a.b = 2
getattr(a, 'b') = 2


**Why use `getattr`?**

1. When the attribute name is in a variable:
   ```python
   attr_name = 'b'
   value = getattr(a, attr_name)  # Same as a.b
   ```

2. When the attribute might not exist (use default):
   ```python
   value = getattr(a, 'missing', 'default')  # Returns 'default' if 'missing' doesn't exist
   ```

3. When you're dynamically deciding which attribute to access:

In [57]:
# =====================================================
# DYNAMIC ATTRIBUTE ACCESS
# =====================================================

# Get either 'a' or 'b' depending on a random condition
# This would be impossible with dot notation!

import random

# If random number > 0.5, get 'b', otherwise get 'a'
attr_name = 'b' if random.random() > 0.5 else 'a'
print(f"Randomly chose attribute: {attr_name}")

value = getattr(a, attr_name)
print(f"Value: {value}")

Randomly chose attribute: a
Value: 1


---

### The `__getattr__` Method

`__getattr__` is called when you try to access an attribute that **doesn't exist** on the object. It's a "fallback" mechanism.

**Important:** `__getattr__` is only called for attributes that are NOT found through normal means. If an attribute exists, `__getattr__` is not called.

In [58]:
# =====================================================
# CLASS WITH __getattr__
# =====================================================

class B:
    """
    A class that responds to ANY attribute access!
    
    - Has real attributes a and b
    - For any OTHER attribute, __getattr__ is called
    """
    
    # Real attributes
    a = 1
    b = 2
    
    def __getattr__(self, k):
        """
        Called when accessing an attribute that doesn't exist.
        
        Arguments:
            k - The name of the attribute being accessed (as a string)
        
        Returns:
            A custom response for missing attributes
        
        Special handling:
            If attribute name starts with '_', raise AttributeError.
            This is important because Python uses many internal attributes
            starting with '_' and we shouldn't interfere with those.
        """
        # Don't handle private/internal attributes (those starting with _)
        if k[0] == '_': 
            raise AttributeError(k)
        
        # For other missing attributes, return a friendly message
        return f'Hello from {k}'

In [59]:
# Create an instance of B
b = B()

In [60]:
# =====================================================
# ACCESSING REAL VS MISSING ATTRIBUTES
# =====================================================

# Access a REAL attribute - __getattr__ is NOT called
print(f"b.a = {b.a}")  # Returns 1 (the actual value)

b.a = 1


In [61]:
# Access a MISSING attribute - __getattr__ IS called
print(f"b.foo = {b.foo}")  # Returns "Hello from foo"

b.foo = Hello from foo


In [62]:
# Try more made-up attributes!
print(f"b.xyz = {b.xyz}")
print(f"b.anything = {b.anything}")
print(f"b.deep_learning = {b.deep_learning}")

b.xyz = Hello from xyz
b.anything = Hello from anything
b.deep_learning = Hello from deep_learning


**Why is this useful?**

1. **Delegation** - Forward attribute access to another object
   ```python
   class Wrapper:
       def __init__(self, wrapped):
           self.wrapped = wrapped
       
       def __getattr__(self, name):
           return getattr(self.wrapped, name)  # Forward to wrapped object
   ```

2. **Lazy loading** - Load data only when first accessed

3. **Dynamic APIs** - Like our callback system that uses `getattr` to call callback methods by name

4. **Proxies and wrappers** - Objects that forward calls to other objects

---

### 🎮 Interactive: the `__getattr__` Lookup Walk

When you write `b.something`, Python walks a fixed path: **instance `__dict__` → class attributes → only then `__getattr__`**. Pick an attribute (`b.a`, `b.foo`, `b._secret`, or type your own) and step through the walk.

**Every flowchart box is clickable** — jump to any checkpoint, and the matching line of `B`'s code lights up. For `b.a` the lower boxes appear dashed: click one and it explains that the walk *stopped early* because lookup ends at the first hit. For `b._secret`, watch the underscore guard `raise` — exactly the failure your fastai `__getattr__` delegation must preserve.

**Controls:** ⏮ / ⏭ step one micro-step at a time · ▶ Play auto-runs it (🐢–🐇 slider sets the speed) · drag the **scrubber** to any step · and **click the highlighted stages** to jump straight there — the matching **code line lights up** and the purple panel explains that exact moment in plain English.

In [63]:
# 📊 Interactive visualization — run this cell to display it inline.
# The HTML file lives in the folder `06_foundations_visuals/`, which must sit
# next to this notebook. If your editor blocks the iframe (e.g. some VS Code
# setups), just open the HTML file directly in your browser instead.
from IPython.display import HTML
HTML("""
<iframe src="06_foundations_visuals/getattr_lookup_flow.html"
        style="width:100%; height:600px; border:1px solid #dde5f2;
               border-radius:12px; box-shadow:0 8px 24px rgba(123,92,214,.12);"
        loading="lazy"></iframe>
<script>addEventListener("message",function(e){if(e.data&&e.data.type==="ce-frame-height"&&e.data.height>50){var fs=document.querySelectorAll("iframe");for(var i=0;i<fs.length;i++){if(fs[i].contentWindow===e.source){fs[i].style.height=e.data.height+"px";break;}}}});</script>
""")

---

# Summary

---

## Key Concepts Learned

### Callbacks
- **What**: Functions/objects that are "called back" at specific points during execution
- **Why**: Customize behavior without modifying the main code
- **In deep learning**: Used for logging, early stopping, checkpointing, etc.

### Lambda Functions
- **Syntax**: `lambda arguments: expression`
- **What**: Anonymous, one-line functions
- **When**: For simple functions needed in one place

### Partial Functions
- **Syntax**: `partial(function, arg1, arg2, ...)`
- **What**: Creates a new function with some arguments pre-filled
- **When**: When you need a function with fewer parameters

### Callable Classes
- **How**: Define `__call__` method
- **What**: Objects that can be called like functions
- **When**: When callbacks need to store state

### `*args` and `**kwargs`
- `*args`: Collects positional arguments into a tuple
- `**kwargs`: Collects keyword arguments into a dictionary
- Also used to unpack when calling functions

### Dunder Methods
- **What**: Special methods like `__init__`, `__add__`, `__repr__`
- **Why**: Customize how Python operators and functions work with your objects
- **Examples**: `__call__` makes objects callable, `__add__` defines `+`

### `getattr` and `__getattr__`
- `getattr(obj, name)`: Built-in function to get attribute by name
- `__getattr__`: Method called when accessing missing attributes

---

## How These Concepts Are Used in Deep Learning

| Concept | Deep Learning Use |
|---------|-------------------|
| Callbacks | Training loop hooks (on_epoch_end, on_batch_end, etc.) |
| Callable classes | Custom layers, loss functions, callbacks with state |
| `*args`, `**kwargs` | Flexible APIs, passing options through layers |
| Dunder methods | Custom tensors, model classes, data containers |
| `getattr` | Dynamic method dispatch in training loops |

These patterns will appear throughout the fastai library and other deep learning frameworks!

---